In [2]:
#######################################
# libs
#

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [3]:
#######################################
# 1. Load dataframe
#

data = pd.read_csv('census_income_dataset.csv')
data.head(2)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gains,capital-loss,hours-per-week,native-country,target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K


In [4]:
#######################################
# 2. Get cols
#

cols = data.keys().tolist()
print(cols)

['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gains', 'capital-loss', 'hours-per-week', 'native-country', 'target']


In [5]:
#######################################
# 3 (1/2). Count nulls
#

count = data.isnull().sum()
display(count)

print(data.shape)
print(type(count))

max = 0;
for key, val in count.items():
    val = val / data.shape[0] * 100
    print(key, '\t', f'{val:.2f}%')
    if val > max:
        max = val

print('max', f'{max:.2f}%')

age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gains        0
capital-loss         0
hours-per-week       0
native-country     583
target               0
dtype: int64

(32561, 15)
<class 'pandas.core.series.Series'>
age 	 0.00%
workclass 	 5.64%
fnlwgt 	 0.00%
education 	 0.00%
education-num 	 0.00%
marital-status 	 0.00%
occupation 	 5.66%
relationship 	 0.00%
race 	 0.00%
sex 	 0.00%
capital-gains 	 0.00%
capital-loss 	 0.00%
hours-per-week 	 0.00%
native-country 	 1.79%
target 	 0.00%
max 5.66%


In [6]:
#######################################
# 3 (2/2). Count nulls
#          'Series' object has no attribute 'insnull'
#

# for col in cols:
#     count = data[col].insnull().sum()
#     print(col, ' nulls:', count)

In [7]:
#######################################
# 4. Count distinct values (to find out `?` values)
#

for col in cols:
    count = data[col].value_counts()
    print('--------', col)
    print(count)

-------- age
age
36    898
31    888
34    886
23    877
35    876
     ... 
83      6
88      3
85      3
86      1
87      1
Name: count, Length: 73, dtype: int64
-------- workclass
workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64
-------- fnlwgt
fnlwgt
164190    13
203488    13
123011    13
148995    12
121124    12
          ..
232784     1
325573     1
140176     1
318264     1
257302     1
Name: count, Length: 21648, dtype: int64
-------- education
education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Presch

In [8]:
#######################################
# 5. Remove non-related columns
#

cols_to_remove = ['fnlwgt', 'education', 'relationship', 'race', 'sex']

data_dropped_cols = data.drop(cols_to_remove, axis=1)
data_dropped_cols.head(2)

,age,workclass,education-num,marital-status,occupation,capital-gains,capital-loss,hours-per-week,native-country,target
0,39,State-gov,13,Never-married,Adm-clerical,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,0,0,13,United-States,<=50K


In [9]:
#######################################
# 6. Get updated cols
#

cols = data_dropped_cols.keys().tolist()
print(cols)

['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'capital-gains', 'capital-loss', 'hours-per-week', 'native-country', 'target']


In [10]:
#######################################
# 7. encode 
#

enc = LabelEncoder()

for col in cols:
    value = data_dropped_cols[col][0]
    
    if isinstance(value, str):
        data_dropped_cols[col] = enc.fit_transform(data_dropped_cols[col].astype('str'))

data_dropped_cols.head(2)

,age,workclass,education-num,marital-status,occupation,capital-gains,capital-loss,hours-per-week,native-country,target
0,39,6,13,4,0,2174,0,40,38,0
1,50,5,13,2,3,0,0,13,38,0


In [11]:
#######################################
# 8. Outliers 
#

for col in cols:
    el = data_dropped_cols[col]
    min_deviation = el.mean() - 3 * el.std()
    max_deviation = el.mean() + 3 * el.std()
    
    X = data_dropped_cols[(el >= min_deviation) & (el <= max_deviation)]
    print(data_dropped_cols.shape[0], X.shape[0], f'Outliers:{(1-X.shape[0]/data_dropped_cols.shape[0]) * 100 : .2f}%', col)

32561 32440 Outliers: 0.37% age
32561 32561 Outliers: 0.00% workclass
32561 32342 Outliers: 0.67% education-num
32561 32561 Outliers: 0.00% marital-status
32561 32561 Outliers: 0.00% occupation
32561 32346 Outliers: 0.66% capital-gains
32561 31091 Outliers: 4.51% capital-loss
32561 32121 Outliers: 1.35% hours-per-week
32561 31448 Outliers: 3.42% native-country
32561 32561 Outliers: 0.00% target


In [12]:
#######################################
# 9. Max outlier percentage is less than 5%. Keep them
#


In [13]:
#######################################
# 10. Normalize/Standardize
#

normalized_data = (data_dropped_cols - data_dropped_cols.mean()) / data_dropped_cols.std()
print(normalized_data.head(2))

        age  workclass  education-num  marital-status  occupation  \
0  0.030670   1.658386       1.134721        0.921620   -1.483559   
1  0.837096   1.026290       1.134721       -0.406206   -0.790497   

   capital-gains  capital-loss  hours-per-week  native-country   target  
0       0.148451     -0.216656       -0.035429        0.251374 -0.56319  
1      -0.145918     -0.216656       -2.222119        0.251374 -0.56319  


In [14]:
#######################################
# 11. Use data_dropped_cols as preprocessed_data
#

preprocessed_data = data_dropped_cols.copy()
preprocessed_data.head(2)

,age,workclass,education-num,marital-status,occupation,capital-gains,capital-loss,hours-per-week,native-country,target
0,39,6,13,4,0,2174,0,40,38,0
1,50,5,13,2,3,0,0,13,38,0


In [15]:
#######################################
# 12. preprocessed_data.to_csv()
#     Do not save preprocessed data with indices!
#

preprocessed_data.to_csv('census_income_dataset_preprocessed.csv', index=False)